# Bedrock Agent 

#### Authenticate with AWS

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv()
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *


for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()


#### Choose the set of metrics for evaluation

In [ ]:
metrics = single_ag_metrics
print("You've chosen the following metrics for evaluating the single Bedrock agent:")
for m in metrics:
    print(f"  - {m}")

#### Select AWS region and Foundation Model 

In [ ]:
region = "us-east-1"
print(f"AWS Region: {region}")

FOUNDATION_MODEL = "anthropic.claude-3-sonnet-20240229-v1:0"
print(f"LLM: {FOUNDATION_MODEL}")

#### Install UAEF 
`pip install uaef`

In [ ]:
from uaef.data import parse_ground_truth_row
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime
print("✓ UAEF imported successfully!")

## Create Bedrock Agent

#### Step 1: Build a simple Bedrock agent

In [ ]:
import boto3
import json
import time
import uuid 
bedrock = boto3.client("bedrock-agent", region_name=region)
iam = boto3.client("iam", region_name=region)

# 1. Create (or reuse) an IAM role for the agent
AGENT_ROLE_NAME = "BedrockAgentRole"
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    role = iam.get_role(RoleName=AGENT_ROLE_NAME)
    role_arn = role["Role"]["Arn"]
    print(f"✓ Reusing existing role: {role_arn}")
except iam.exceptions.NoSuchEntityException:
    role = iam.create_role(
        RoleName=AGENT_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for Bedrock Agent UAEF demo"
    )
    role_arn = role["Role"]["Arn"]
    # Attach the Bedrock full-access managed policy (for demo purposes)
    iam.attach_role_policy(
        RoleName=AGENT_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/AmazonBedrockFullAccess"
    )
    print(f"✓ Created role: {role_arn}")
    print("  Waiting 10s for IAM propagation...")
    time.sleep(10)

# 2. Create the agent  
AGENT_NAME = f"demo-bedrock-agent-{uuid.uuid4().hex[:8]}"

response = bedrock.create_agent(
    agentName=AGENT_NAME,
    agentResourceRoleArn=role_arn,
    foundationModel=FOUNDATION_MODEL,
    instruction=(
        "You are a helpful assistant. Answer user questions clearly and concisely. "
        "Use available tools when they are relevant, but you can always answer general "
        "If you don't know the answer, say so."
    ),
    idleSessionTTLInSeconds=600,
    description="Demo agent for UAEF evaluation"
)

agent_id = response["agent"]["agentId"]
print(f"✓ Created agent: {agent_id}")

# 3. Wait for agent to finish creating, then prepare it
print("  Waiting for agent to be ready for preparation...")
for _ in range(30):
    time.sleep(5)
    status = bedrock.get_agent(agentId=agent_id)["agent"]["agentStatus"]
    print(f"  Status: {status}")
    if status == "NOT_PREPARED":
        break

bedrock.prepare_agent(agentId=agent_id)
print("  Preparing agent...")

# Wait for PREPARED status
for _ in range(30):
    time.sleep(5)
    status = bedrock.get_agent(agentId=agent_id)["agent"]["agentStatus"]
    print(f"  Status: {status}")
    if status == "PREPARED":
        break

# 4. Create an alias to invoke it
alias_response = bedrock.create_agent_alias(
    agentId=agent_id,
    agentAliasName="live"
)
alias_id = alias_response["agentAlias"]["agentAliasId"]
print(f"✓ Created alias: {alias_id}")

# Wait for alias to be ready
time.sleep(5)

print(f"\nagent_id  = '{agent_id}'")
print(f"alias_id = '{alias_id}'")


#### Step 2: Run the Bedrock Agent

In [ ]:
import boto3
import uuid

bedrock_runtime = boto3.client("bedrock-agent-runtime", region_name="us-east-1")

session_id = str(uuid.uuid4())
user_input = "What are the benefits of using microservices architecture?"



response = bedrock_runtime.invoke_agent(
    agentId=agent_id,
    agentAliasId=alias_id,
    sessionId=session_id,
    inputText=user_input
)

# Consume the EventStream into a list
event_stream = list(response["completion"])

for i, event in enumerate(event_stream):
    if "chunk" in event:
        text = event["chunk"]["bytes"].decode("utf-8")
        print(f"[chunk {i}] {text[:200]}...")
    elif "trace" in event:
        print(f"[trace {i}] keys: {list(event['trace'].get('trace', {}).keys())}")
    else:
        print(f"[event {i}] keys: {list(event.keys())}")


#### Step 3: Use Bedrock Adapter 

In [ ]:
from uaef.adapters import BedrockAgentAdapter
from uaef.api import evaluate
from uaef.models import GroundTruth

# Package the data for the BedrockAgentAdapter 
bedrock_data = {
    "event_stream": event_stream,
    "user_input": user_input,
    "session_id": session_id,
    "agent_id": agent_id,
    "alias_id": alias_id,
}

# Transform to AgentTrace
adapter = BedrockAgentAdapter()
agent_trace = adapter.transform_to_canonical(bedrock_data)

print(f"✓ Transformed Bedrock output to AgentTrace")
print(f"  Trace ID:   {agent_trace.trace_id}")
print(f"  Messages:   {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")
print(f"  Framework:  {agent_trace.framework}")

# Print the agent's response
for msg in agent_trace.messages:
    if msg.role.value == "assistant":
        print(f"\nAgent Response:\n{msg.content[:500]}")



#### Step 4: Evaluate

In [ ]:
# Evaluate
ground_truth = GroundTruth(
    expected_output="Microservices provide benefits like independent deployment, scalability, and technology flexibility",
    context_documents=["Microservices architecture decomposes applications into small, independent services"]
)

result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics
)

print(f"\n{'='*50}")
print("BEDROCK AGENT EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        print(f"  {metric.metric_name}: {metric.score:.2f}")


## Batch Evaluation
1. Modify `data/ground-truth.xlsx` with your Ground Truth questions and answers.
2. Run batch evaluation by sending queries from Ground Truth to the live Bedrock agent. 

#### Load Ground Truth Q & A

In [ ]:
import pandas as pd
import json
from datetime import datetime, timezone
from uuid import uuid4

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole

# Load ground truth data from Excel
excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)

# Convert Excel rows to JSON records for inspection
gt_json = df.to_dict(orient="records")

print(f"✓ Converted {len(gt_json)} rows to JSON")
print(f"✓ Loaded {len(df)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()



#### Reuse Bedrock Agent

In [ ]:
# Reuse the Bedrock agent and adapter from Steps 1-3 
import uuid
from uaef.adapters import BedrockAgentAdapter

adapter = BedrockAgentAdapter()
traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query, expected, context, expected_tools = parse_ground_truth_row(row)

    # Send query to the Bedrock agent
    session_id = str(uuid.uuid4())
    response = bedrock_runtime.invoke_agent(
        agentId=agent_id,
        agentAliasId=alias_id,
        sessionId=session_id,
        inputText=query
    )
    event_stream = list(response["completion"])

    # Package for BedrockAgentAdapter
    bedrock_data = {
        "event_stream": event_stream,
        "user_input": query,
        "session_id": session_id,
        "agent_id": agent_id,
        "alias_id": alias_id,
    }

    trace = adapter.transform_to_canonical(bedrock_data)
    traces.append(trace)
    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else []
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}")

print(f"\n✓ Ran {len(traces)} queries through the Bedrock agent")

#### Run evaluation on the agent traces

In [ ]:
# Batch evaluate all traces
batch_results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    metrics=metrics,
    max_workers=4
)

print(f"{'='*50}")
print(f"BATCH RESULTS — BEDROCK AGENT")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

#### Export batch evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="bedrock_batch_results")
